# Cleaning Pipeline 03 — Near-Duplicate Detection with Perceptual Hashing

Perceptual hashing summarizes visual structure instead of raw bytes. It can detect resized, recompressed, mildly color-adjusted, or format-converted copies.

Two pHashes are compared with Hamming distance: `0` is closest; small values are likely near-duplicates.

This notebook uses five-band candidate generation to keep the search practical. With a Hamming threshold of 4, five bands guarantee that at least one band is unchanged for every true pair within the threshold.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "cleaning_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))
DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_ROOT = REPO_ROOT / "eda_outputs" / "notebook_pipeline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
from collections import defaultdict
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash
from tqdm.auto import tqdm

manifest_path = OUTPUT_ROOT / "00_manifest" / "train_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Run notebook 00 first: {manifest_path}")
manifest = pd.read_csv(manifest_path)
valid = manifest[manifest["is_valid"]].copy().reset_index(drop=True)

In [ ]:
HASH_SIZE = 8
DISTANCE_THRESHOLD = 4
NUM_BANDS = 5

def compute_phash(path: str):
    with Image.open(path) as img:
        return str(imagehash.phash(ImageOps.exif_transpose(img).convert("RGB"), hash_size=HASH_SIZE))

valid["phash"] = [compute_phash(path) for path in tqdm(valid["path"], desc="Computing pHash")]
display(valid[["relative_path", "phash"]].head())

In [ ]:
hash_hex_len = len(valid["phash"].iloc[0])
band_width = hash_hex_len // NUM_BANDS
buckets = [defaultdict(list) for _ in range(NUM_BANDS)]

for idx, hash_value in enumerate(valid["phash"]):
    for band in range(NUM_BANDS):
        start = band * band_width
        end = hash_hex_len if band == NUM_BANDS - 1 else (band + 1) * band_width
        buckets[band][hash_value[start:end]].append(idx)

candidate_pairs = set()
for band_buckets in buckets:
    for indices in band_buckets.values():
        if len(indices) < 2:
            continue
        for i, j in itertools.combinations(indices, 2):
            candidate_pairs.add((min(i, j), max(i, j)))

print("Candidate pairs after banding:", len(candidate_pairs))

In [ ]:
pairs = []
for i, j in tqdm(candidate_pairs, desc="Exact pHash distances"):
    hash_i = imagehash.hex_to_hash(valid.loc[i, "phash"])
    hash_j = imagehash.hex_to_hash(valid.loc[j, "phash"])
    distance = int(hash_i - hash_j)
    if distance <= DISTANCE_THRESHOLD:
        pairs.append({
            "path_a": valid.loc[i, "path"],
            "relative_path_a": valid.loc[i, "relative_path"],
            "class_a": valid.loc[i, "class_name"],
            "label_a": int(valid.loc[i, "label"]),
            "path_b": valid.loc[j, "path"],
            "relative_path_b": valid.loc[j, "relative_path"],
            "class_b": valid.loc[j, "class_name"],
            "label_b": int(valid.loc[j, "label"]),
            "phash_distance": distance,
            "cross_label": int(valid.loc[i, "label"]) != int(valid.loc[j, "label"]),
        })

phash_pairs = pd.DataFrame(pairs)
if len(phash_pairs):
    phash_pairs = phash_pairs.sort_values(["cross_label","phash_distance"], ascending=[False,True])
print("Near-duplicate pairs:", len(phash_pairs))
print("Cross-label pairs:", int(phash_pairs["cross_label"].sum()) if len(phash_pairs) else 0)
display(phash_pairs.head(30))

In [ ]:
def show_pairs(df, n=12, title="pHash candidate pairs"):
    sample = df.head(n)
    if len(sample) == 0:
        print("No pairs to display"); return
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, len(sample) * 3.0))
    axes = np.asarray(axes).reshape(len(sample), 2)
    for row_idx, (_, row) in enumerate(sample.iterrows()):
        for col_idx, side in enumerate(["a","b"]):
            path = row[f"path_{side}"]
            with Image.open(path) as img:
                axes[row_idx, col_idx].imshow(ImageOps.exif_transpose(img).convert("RGB"))
            axes[row_idx, col_idx].axis("off")
            axes[row_idx, col_idx].set_title(
                f"{Path(path).name} | {row[f'class_{side}']}\n"
                f"distance={row['phash_distance']} cross_label={row['cross_label']}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_pairs(phash_pairs, title="Closest pHash pairs")

## Limitations

pHash can miss large crops, viewpoint changes, and semantic similarity. It can also produce false positives for simple images with similar layouts. Keep these as manual-review candidates unless they are also exact duplicates.

In [ ]:
stage_dir = OUTPUT_ROOT / "03_phash"
stage_dir.mkdir(parents=True, exist_ok=True)
valid[["path","relative_path","class_name","label","phash"]].to_csv(stage_dir / "image_phash.csv", index=False)
phash_pairs.to_csv(stage_dir / "phash_duplicate_pairs.csv", index=False)
print("Saved outputs to", stage_dir)